# X-Ray AI Training Project
### Database Download, editing and organizing

In [2]:
import os
import shutil
import random
from google.colab import userdata

# =====================================================================
# 1. ENVIRONMENT SETUP & AUTHENTICATION
# =====================================================================
# Retrieve credentials securely from Colab's Secret Manager
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

print("📥 Downloading dataset via Kaggle API...")
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia > /dev/null
!unzip -q chest-xray-pneumonia.zip -d dataset/
print("✅ Dataset downloaded and extracted.")

# =====================================================================
# 2. DATASET RESTRUCTURING (FIXING ORIGINAL SPLIT)
# =====================================================================
# The original dataset has a flawed validation split (only 16 images).
# We merge train/val and recreate a robust 80/20 split.
base_dir = '/content/dataset/chest_xray'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
classes = ['NORMAL', 'PNEUMONIA']

print("🔄 Restructuring data for an 80/20 train/validation split...")

for cls in classes:
    train_imgs = os.listdir(os.path.join(train_dir, cls))
    val_imgs = os.listdir(os.path.join(val_dir, cls))
    all_imgs = train_imgs + val_imgs
    random.shuffle(all_imgs)

    split_idx = int(len(all_imgs) * 0.8)
    new_train, new_val = all_imgs[:split_idx], all_imgs[split_idx:]

    # Move files to proper directories
    for img in val_imgs:
        shutil.move(os.path.join(val_dir, cls, img), os.path.join(train_dir, cls, img))
    for img in new_val:
        shutil.move(os.path.join(train_dir, cls, img), os.path.join(val_dir, cls, img))

print("✅ Data splits fixed.")

📥 Downloading dataset via Kaggle API...
100% 2.29G/2.29G [00:20<00:00, 117MB/s] 
✅ Dataset downloaded and extracted.
🔄 Restructuring data for an 80/20 train/validation split...
✅ Data splits fixed.


### Model Setup and data loaders

In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# =====================================================================
# 3. DATALOADERS & AUGMENTATION (RUN ONCE PER SESSION)
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10), # Prevent overfitting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

batch_size = 32
train_loader = DataLoader(datasets.ImageFolder(train_dir, transform=train_transforms), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(datasets.ImageFolder(val_dir, transform=val_transforms), batch_size=batch_size, shuffle=False)

print("✅ DataLoaders created successfully. No need to run this cell again for new experiments.")

✅ DataLoaders created successfully. No need to run this cell again for new experiments.


### Train-Loader and Val-Loader

In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Definir Transformaciones (Data Augmentation y Normalización para ResNet)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10), # Pequeña rotación para evitar overfitting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Cargar Datasets desde las carpetas
train_dir = '/content/dataset/chest_xray/train'
val_dir = '/content/dataset/chest_xray/val'

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)

# 3. Crear los DataLoaders (¡Esto es lo que faltaba!)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"✅ DataLoaders creados con éxito.")
print(f"Lotes de entrenamiento: {len(train_loader)} (de {batch_size} imágenes cada uno)")
print(f"Lotes de validación: {len(val_loader)}")

✅ DataLoaders creados con éxito.
Lotes de entrenamiento: 131 (de 32 imágenes cada uno)
Lotes de validación: 33


### Experiment parameters

In [7]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# =====================================================================
# 🎛️ 4. EXPERIMENT CONTROL PANEL (CHANGE THIS BETWEEN TRAININGS)
# =====================================================================
# Modify these variables for each new experiment
EXPERIMENT_NAME = "expC_1_clinical_balance"
WEIGHT_NORMAL = 1.0
WEIGHT_PNEUMONIA = 1.2

# ---------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🧪 Preparing: {EXPERIMENT_NAME} | Weights -> Normal: {WEIGHT_NORMAL}, Pneumonia: {WEIGHT_PNEUMONIA}")

# 1. Load fresh pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. Freeze base layers
for param in model.parameters():
    param.requires_grad = False

# 3. Replace classification head for a clean start
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2).to(device)
model = model.to(device)

# 4. Initialize Optimizer & Loss Function with current experiment weights
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
class_weights = torch.FloatTensor([WEIGHT_NORMAL, WEIGHT_PNEUMONIA]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print("✅ Model reset and compiled with new weights. Ready to start training!")

🧪 Preparing: expC_1_clinical_balance | Weights -> Normal: 1.0, Pneumonia: 1.5
✅ Model reset and compiled with new weights. Ready to start training!


### Training loop

In [9]:
import time
import numpy as np
from sklearn.metrics import confusion_matrix
from google.colab import files

# =====================================================================
# 5. TRAINING LOOP WITH BUSINESS METRICS EVALUATION
# =====================================================================
num_epochs = 5
print(f"🚀 Starting training for {num_epochs} epochs...")
start_time = time.time()

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 20)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = val_loader

        running_loss = 0.0
        all_preds, all_labels = [], []

        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        epoch_loss = running_loss / len(dataloader.dataset)

        # Calculate Confusion Matrix for business logic evaluation
        cm = confusion_matrix(all_labels, all_preds)
        tn, fp, fn, tp = cm.ravel()
        accuracy = (tp + tn) / (tp + tn + fp + fn)

        print(f'[{phase.upper()}] Loss: {epoch_loss:.4f} | Accuracy: {accuracy:.4f}')

        if phase == 'val':
            print(f'   🚨 False Negatives (Missed Patients): {fn}')
            print(f'   💸 False Positives (Healthy but flagged): {fp}')
            print(f'   ✅ Correct Predictions: {tp} Pneumonia, {tn} Normal')

total_time = time.time() - start_time
print(f'\n🏁 Training completed in {total_time // 60:.0f}m {total_time % 60:.0f}s')

# Save model artifacts
model_filename = f"{EXPERIMENT_NAME}.pth"
torch.save(model.state_dict(), model_filename)
files.download(model_filename)
print(f"💾 Model weights saved locally as: {model_filename}")


🚀 Starting training for 5 epochs...

Epoch 1/5
--------------------
[TRAIN] Loss: 0.1311 | Accuracy: 0.9348
[VAL] Loss: 0.1158 | Accuracy: 0.9589
   🚨 False Negatives (Missed Patients): 24
   💸 False Positives (Healthy but flagged): 19
   ✅ Correct Predictions: 753 Pneumonia, 251 Normal

Epoch 2/5
--------------------
[TRAIN] Loss: 0.1273 | Accuracy: 0.9384
[VAL] Loss: 0.1167 | Accuracy: 0.9608
   🚨 False Negatives (Missed Patients): 26
   💸 False Positives (Healthy but flagged): 15
   ✅ Correct Predictions: 751 Pneumonia, 255 Normal

Epoch 3/5
--------------------
[TRAIN] Loss: 0.1158 | Accuracy: 0.9505
[VAL] Loss: 0.1246 | Accuracy: 0.9522
   🚨 False Negatives (Missed Patients): 39
   💸 False Positives (Healthy but flagged): 11
   ✅ Correct Predictions: 738 Pneumonia, 259 Normal

Epoch 4/5
--------------------
[TRAIN] Loss: 0.1169 | Accuracy: 0.9479
[VAL] Loss: 0.1104 | Accuracy: 0.9580
   🚨 False Negatives (Missed Patients): 15
   💸 False Positives (Healthy but flagged): 29
   ✅ Cor

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

💾 Model weights saved locally as: expC_1_clinical_balance.pth


### Threshold Tuning

In [11]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import confusion_matrix

# =====================================================================
# 🧪 EXPERIMENT D: THRESHOLD TUNING (DECISION BOUNDARY CALIBRATION)
# =====================================================================
# We evaluate the existing model in memory without retraining.

CUSTOM_THRESHOLD = 0.20  # <-- Adjust here: Predict Pneumonia if confidence > 30%

print(f"🔍 Evaluating validation set with Pneumonia Threshold at {CUSTOM_THRESHOLD*100}%...")

model.eval() # Strict evaluation mode
all_labels = []
all_probs_pneumonia = []

# Disable gradient calculation for faster inference and reduced memory usage
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        # Convert raw logits to probabilities (0.0 to 1.0) using Softmax
        probs = F.softmax(outputs, dim=1)

        # Extract only the probability of Class 1 (Pneumonia)
        probs_pneumonia = probs[:, 1]

        all_probs_pneumonia.extend(probs_pneumonia.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Apply the custom business logic (Thresholding)
# If probability >= threshold, predict 1 (Pneumonia), else 0 (Normal)
custom_predictions = (np.array(all_probs_pneumonia) >= CUSTOM_THRESHOLD).astype(int)

# Calculate the new confusion matrix based on the custom predictions
cm = confusion_matrix(all_labels, custom_predictions)
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("-" * 40)
print(f"📊 RESULTS WITH THRESHOLD {CUSTOM_THRESHOLD}")
print(f"Overall Accuracy: {accuracy:.4f}")
print(f"   🚨 False Negatives (Missed Patients): {fn}")
print(f"   💸 False Positives (Healthy but flagged): {fp}")
print(f"   ✅ Correct Predictions: {tp} Pneumonia, {tn} Normal")
print("-" * 40)

🔍 Evaluating validation set with Pneumonia Threshold at 20.0%...
----------------------------------------
📊 RESULTS WITH THRESHOLD 0.2
Overall Accuracy: 0.9484
   🚨 False Negatives (Missed Patients): 7
   💸 False Positives (Healthy but flagged): 47
   ✅ Correct Predictions: 770 Pneumonia, 223 Normal
----------------------------------------


### Test-Time Augmentation and consensus

In [12]:
import torch
import torch.nn.functional as F
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

# =====================================================================
# 🧪 EXPERIMENT E: TEST-TIME AUGMENTATION (TTA) & MAJORITY VOTING
# =====================================================================
print("🚀 Preparing TTA transforms and loaders...")

# 1. Define the 3 specific transformations
transform_orig = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_rot = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation((10, 10)), # Fixed 10 degrees rotation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_zoom = transforms.Compose([
    transforms.Resize((250, 250)),
    transforms.CenterCrop(224), # Center crop creates a "zoom in" effect
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Create datasets and dataloaders (shuffle=False is CRUCIAL here)
ds_orig = datasets.ImageFolder(val_dir, transform=transform_orig)
ds_rot = datasets.ImageFolder(val_dir, transform=transform_rot)
ds_zoom = datasets.ImageFolder(val_dir, transform=transform_zoom)

# Batch size can be 32. We don't shuffle so patient images align perfectly in the zip loop
dl_orig = DataLoader(ds_orig, batch_size=32, shuffle=False)
dl_rot = DataLoader(ds_rot, batch_size=32, shuffle=False)
dl_zoom = DataLoader(ds_zoom, batch_size=32, shuffle=False)

print("🔍 Running predictions on 3 versions of each image...")
model.eval()
all_labels = []
tta_predictions = []

with torch.no_grad():
    # Zip allows us to iterate through the 3 loaders simultaneously
    for (inputs_o, labels), (inputs_r, _), (inputs_z, _) in zip(dl_orig, dl_rot, dl_zoom):

        inputs_o = inputs_o.to(device)
        inputs_r = inputs_r.to(device)
        inputs_z = inputs_z.to(device)
        labels = labels.to(device)

        # Get predictions for all 3 variations
        out_o = model(inputs_o)
        out_r = model(inputs_r)
        out_z = model(inputs_z)

        # Get binary predictions (0 or 1) using standard 50% threshold for each vote
        _, pred_o = torch.max(out_o, 1)
        _, pred_r = torch.max(out_r, 1)
        _, pred_z = torch.max(out_z, 1)

        # 3. Hard Voting Logic (Consensus)
        # Sum the votes (each is 0 or 1). If sum >= 2, the final prediction is 1 (Pneumonia)
        sum_votes = pred_o + pred_r + pred_z
        final_preds = (sum_votes >= 2).int()

        tta_predictions.extend(final_preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 4. Calculate final metrics
cm = confusion_matrix(all_labels, tta_predictions)
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("-" * 40)
print("📊 RESULTS WITH TTA & MAJORITY VOTING")
print(f"Overall Accuracy: {accuracy:.4f}")
print(f"   🚨 False Negatives (Missed Patients): {fn}")
print(f"   💸 False Positives (Healthy but flagged): {fp}")
print(f"   ✅ Correct Predictions: {tp} Pneumonia, {tn} Normal")
print("-" * 40)

🚀 Preparing TTA transforms and loaders...
🔍 Running predictions on 3 versions of each image...
----------------------------------------
📊 RESULTS WITH TTA & MAJORITY VOTING
Overall Accuracy: 0.9723
   🚨 False Negatives (Missed Patients): 6
   💸 False Positives (Healthy but flagged): 23
   ✅ Correct Predictions: 771 Pneumonia, 247 Normal
----------------------------------------
